# Fine-tuning do Assistente Virtual Médico (LoRA/QLoRA)

Rode este notebook em um runtime com GPU (**Runtime > Change runtime type > GPU**).
Ele reproduz `finetuning/train.py` e `finetuning/evaluate.py` do repositório.

**Antes de começar — token da Hugging Face (recomendado):**
sem ele os downloads são anônimos e compartilham o limite de taxa do IP do Colab,
ficando mais lentos e podendo falhar com HTTP 429 no meio dos 2 GB de pesos.

1. Crie um token de leitura em https://huggingface.co/settings/tokens
2. No Colab, abra **Secrets** (ícone 🔑 na barra lateral)
3. Adicione um secret chamado `HF_TOKEN` com esse valor
4. Ative **Notebook access** para ele

Os scripts detectam esse secret automaticamente. Para modelos de licença
restrita (Llama 3, Gemma), o token é obrigatório e é preciso aceitar os
termos na página do modelo antes.

In [ ]:
# Confirme que o runtime é GPU (Runtime > Change runtime type > T4 GPU).
# TPU não funciona: o QLoRA usa bitsandbytes, que só tem kernels CUDA/ROCm.
!nvidia-smi || echo 'SEM GPU — troque o runtime para T4 GPU antes de continuar.'

In [ ]:
!git clone https://github.com/RenanAmaral/FIAP-9IADT-medical-agent-fine-tuned.git
%cd FIAP-9IADT-medical-agent-fine-tuned
!pip install -q -r requirements.txt

In [ ]:
# Confere se o token foi encontrado (não imprime o valor do token).
from finetuning.hf_auth import ensure_hf_login

ensure_hf_login()

In [ ]:
# O dataset já vem versionado em data/processed/.
# Rode esta célula apenas se quiser regenerá-lo do zero.
!python -m preprocessing.run_pipeline

## Treino

O script imprime o plano de treino antes de começar (exemplos, batch efetivo,
número de atualizações de peso) e avisa se forem poucas demais para produzir
um modelo mensuravelmente diferente do base.

Com os padrões atuais são ~168 atualizações, e o treino leva poucos minutos
numa T4.

In [ ]:
!python -m finetuning.train --base-model TinyLlama/TinyLlama-1.1B-Chat-v1.0

In [ ]:
!python -m finetuning.evaluate --base-model TinyLlama/TinyLlama-1.1B-Chat-v1.0 --adapter-dir finetuning/adapters/medical-assistant-lora --test-file data/processed/test.jsonl

In [ ]:
# Relatório de avaliação (perplexidade, ROUGE e comparação antes/depois)
from IPython.display import Markdown, display

display(Markdown(open('finetuning/eval_results/evaluation_report.md').read()))

In [ ]:
# Teste rápido do assistente completo já com o modelo fine-tuned
!python -m assistant.database
!python -m graphs.cli --backend finetuned --paciente PAC-0003 --pergunta "Qual a conduta para este paciente?"

In [ ]:
# Baixa os adapters e o relatório de avaliação para a máquina local
!zip -r resultados_finetuning.zip finetuning/adapters finetuning/eval_results
from google.colab import files

files.download('resultados_finetuning.zip')

## Próximo passo

Descompacte `resultados_finetuning.zip` na raiz do repositório local e commite
`finetuning/eval_results/`. Depois preencha a tabela de métricas do §3.6 de
`docs/relatorio_tecnico.md` com os números obtidos aqui.